# Random Forest / XGBoost Classifier

In [ ]:
import pandas as pd
import numpy as np
import mlflow
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import sys
sys.path.append('..')
from src.data_utils import stratified_split

## Dados

In [ ]:
print("A carregar dados...")
df = pd.read_csv('../data/dataset_final.csv')

    # 90/10
temp_df, test_df = stratified_split(df, 'text', 'label', test_size=0.10, seed=2026)

# Dos 90% que sobraram, tirar ~11% para validação
    # (o que equivale a 10% do dataset original)
train_df, val_df = stratified_split(temp_df, 'text', 'label', test_size=0.1111, seed=2026)

print(f"Tamanhos - Treino: {len(train_df)} | Validação: {len(val_df)} | Teste: {len(test_df)}")

## Encoding

In [ ]:
print("A codificar as labels para números (para o XGBoost não se queixar)...")
le = LabelEncoder()
y_train = le.fit_transform(train_df['label'])
y_val = le.transform(val_df['label'])
y_test = le.transform(test_df['label'])

# Guardar o mapeamento
classes_nomes = le.classes_
print("Mapeamento das classes:", {i: label for i, label in enumerate(classes_nomes)})

## TF-IDF

In [ ]:
MAX_FEATURES = 20000
tfidf = TfidfVectorizer(max_features=MAX_FEATURES)

X_train = tfidf.fit_transform(train_df['text'])
X_val = tfidf.transform(val_df['text'])
X_test = tfidf.transform(test_df['text'])

print("TF-IDF aplicado")

## Treino

### Random Forest

In [ ]:
modelo_rf = RandomForestClassifier(n_estimators=100, random_state=2026, n_jobs=-1)
modelo_rf.fit(X_train, y_train)

# 4. Avaliar
preds = modelo_rf.predict(X_val)
acc = accuracy_score(y_val, preds)
print(f"Accuracy Real na Validação: {acc * 100:.2f}%")
print(classification_report(y_val, preds))

### XGBoost

In [ ]:
parametros_xgb = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 6, 9],
    'learning_rate': [0.05, 0.1, 0.2]
}

print("A procurar os melhores parâmetros para o XGBoost...")
xgb_base = XGBClassifier(random_state=2026, n_jobs=-1)
search = RandomizedSearchCV(
    estimator=xgb_base, 
    param_distributions=parametros_xgb, 
    n_iter=5, # Testa 5 combinações diferentes
    cv=3,     # Validação cruzada com 3 folds
    scoring='accuracy', 
    verbose=2, 
    random_state=2026
)

# mlflow
with mlflow.start_run(run_name="XGBoost_Tuned"):

    search.fit(X_train, y_train)
    
    melhor_modelo = search.best_estimator_
    
        # Prever na Validação e no Teste
    preds_val = melhor_modelo.predict(X_val)
    preds_test = melhor_modelo.predict(X_test)
    
    acc_val = accuracy_score(y_val, preds_val)
    acc_test = accuracy_score(y_test, preds_test)
    
        # Registar tudo no MLflow automaticamente
    mlflow.log_param("max_features", MAX_FEATURES)
    mlflow.log_params(search.best_params_)
    mlflow.log_metric("val_acc", acc_val)
    mlflow.log_metric("test_acc", acc_test)
    
    print("\n" + "="*40)
    print(f"MELHORES PARÂMETROS: {search.best_params_}")
    print(f"ACCURACY VALIDAÇÃO: {acc_val * 100:.2f}%")
    print(f"ACCURACY TESTE (FINAL): {acc_test * 100:.2f}%")
    print("="*40 + "\n")
    
    print("Classification Report do TESTE:")
    print(classification_report(y_test, preds_test, target_names=classes_nomes))
    
    # Guardar os modelos
    import joblib
    joblib.dump(melhor_modelo, '../modelos/xgboost_tuned.pkl')
    joblib.dump(tfidf, '../modelos/tfidf_xgb.pkl')
    joblib.dump(le, '../modelos/label_encoder_xgb.pkl')
    print("Modelos guardados na pasta 'modelos/'")